In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training/tasks/esm2-enzyme-transfer")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # that holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "helpers").is_dir() and (folder / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with helpers/ and checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 04 Evaluate both models on every species

**What this step is for:** find out whether fine-tuning really helped, and where. You score the floor, the baseline, and the fine-tuned model on the test proteins of all five species, measure how sure each score is, test whether the fine-tuned model beats the baseline by more than chance, look at which classes each model mixes up, and make the headline figure of the project.

**Time:** about 1.25 hours. Right after this notebook you have check-in 2 with your mentor.

**Reads:** `predictions_baseline.tsv` from notebook 02 and `predictions_finetuned.tsv` from notebook 03, in your run folder on Google Drive, and the divergence times in `data/divergence_times.tsv`. **Writes:** `results.tsv`, `paired_differences.tsv`, `per_class_f1.tsv`, `figures/confusion_grid.png`, and `figures/distance_curve.png` in the same folder.

**Before you start:** this notebook needs no GPU; a normal (CPU) runtime is fine. Run the Setup cell at the top and work top to bottom. If a cell below a `TODO` shows an error, first make sure that `TODO` passed its check.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                # a label for this run: the small ESM-2 model, with 8 million parameters
WORK_DIR = setup_workdir(RUN_NAME)  # reconnects Google Drive on Colab and returns the run folder
print(WORK_DIR)

In [ ]:
# The standard data tools (notebook 00, Part 4), and the macro-F1 score from notebook 02.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

## Load both models' predictions
Notebooks 02 and 03 each saved a table with one row per test protein. They share three columns: `accession`, `species`, and `y_true` (the real class). Each also has its own prediction columns: `y_pred_floor` and `y_pred_baseline` from notebook 02, and `y_pred_finetuned` from notebook 03.

`merge` puts two tables side by side, matching up the rows that agree on the columns you list (like a join in a database). The result, `preds`, has one row per test protein and one prediction column for each of the three models.

In [ ]:
# require_file stops with a clear message if the notebook that makes a file has not saved it yet.
baseline_path = require_file(WORK_DIR / "predictions_baseline.tsv", made_by="Notebook 02 (baseline)")
finetuned_path = require_file(WORK_DIR / "predictions_finetuned.tsv", made_by="Notebook 03 (fine-tune)")
baseline_preds = pd.read_csv(baseline_path, sep="\t")
finetuned_preds = pd.read_csv(finetuned_path, sep="\t")

# Match rows on the three shared columns. A protein appears in preds only if both files have it.
preds = baseline_preds.merge(finetuned_preds, on=["accession", "species", "y_true"])
print(f"baseline rows: {len(baseline_preds):,}   fine-tuned rows: {len(finetuned_preds):,}   merged rows: {len(preds):,}")
preds.head()

In [ ]:
# How many test proteins each species has, in our usual order.
preds["species"].value_counts().loc[SPECIES_ORDER]

## Macro-F1 for one species
A reminder from notebook 02: for one class, **F1** is high only when the model both finds most of the proteins of that class (recall) and is usually right when it names that class (precision). **Macro-F1** is the plain average of the F1 of every class, so a rare class counts as much as a common one. It runs from 0 (useless) to 1 (perfect).

Start with one species, yeast. Notebook 03 already did this for human; here you do it for both models.

In [ ]:
# TODO: Compute the macro-F1 of the baseline and of the fine-tuned model on the yeast proteins only.
#       Store them in yeast_baseline_f1 and yeast_finetuned_f1.
# Hint: keep the yeast rows of preds (notebook 00, Part 7), then call f1_score(true_classes, predicted_classes,
#       average="macro", zero_division=0) once with the y_pred_baseline column and once with the y_pred_finetuned column.
yeast_baseline_f1 = None  # replace None with your code
yeast_finetuned_f1 = None  # replace None with your code

In [ ]:
check_species_f1(yeast_baseline_f1, preds, "yeast", "baseline", name="yeast_baseline_f1");
check_species_f1(yeast_finetuned_f1, preds, "yeast", "finetuned", name="yeast_finetuned_f1");

In [ ]:
print(f"Yeast macro-F1: baseline {yeast_baseline_f1:.3f}   fine-tuned {yeast_finetuned_f1:.3f}")

## Every species, with error bars
One score from one set of test proteins is not the whole story. If we had happened to test on a slightly different set of yeast proteins, the score would come out a little different. How much could it move? A **bootstrap** answers that with the data we already have:

1. Re-draw the test proteins of one species at random, as many as there are, **with replacement**: like drawing names from a hat and putting each name back after you write it down. Some proteins get drawn twice, and some not at all.
2. Score the model on that re-drawn set.
3. Repeat 1,000 times. You now have 1,000 scores, and they spread out around the real one.

The **95% interval** is the range that holds the middle 95% of those 1,000 scores (it leaves out the lowest 2.5% and the highest 2.5%). A narrow interval means the score barely moves when the test proteins change, so we can trust it. A wide interval means it could easily have come out quite different.

Our helper `score_table(preds)` does this for every species and every model (floor, baseline, and fine-tuned). It returns a table with one row per species and model, and these columns: `species`, `model`, `macro_f1`, `accuracy`, and `ci_low` and `ci_high`, the two ends of the 95% interval ("CI" is short for confidence interval). It runs 15,000 scorings, so it takes a few seconds.

**Prediction (write this before you run the next cell):** Rank the five species (human, mouse, fly, yeast, ecoli) from the highest fine-tuned macro-F1 to the lowest. Why that order?

In [ ]:
# TODO: Score every species and model, with bootstrap intervals. Store the table in results.
# Hint: use score_table(preds)
results = None  # replace None with your code

In [ ]:
check_results_table(results, preds);

In [ ]:
results.round(3)  # round to 3 decimal places so the table is easier to read

In [ ]:
# The same macro-F1 values as a grid: one row per species, one column per model.
# pivot reshapes a long table into a grid; .loc puts the rows and columns in the order we list.
results.pivot(index="species", columns="model", values="macro_f1").loc[SPECIES_ORDER, ["floor", "baseline", "finetuned"]].round(3)

**Compare:** Was your prediction right? What surprised you? Also look back at the last prediction you wrote in notebook 01: which test species did you expect the model to do worst on, and why? Were you right, and does your reason still hold up?

## Did fine-tuning really help? The paired interval
Each species now has two macro-F1 scores, one for the baseline and one for the fine-tuned model. Is the difference between them real, or could it come from the luck of which proteins we happened to test?

Comparing the two separate intervals is not the best test. Both models are scored on the *same* proteins, and some proteins are hard for both of them. A **paired bootstrap** uses that:

1. Re-draw the test proteins of one species with replacement, as before.
2. Score **both** models on that same re-drawn set, and record the difference: fine-tuned minus baseline.
3. Repeat 1,000 times, and take the middle 95% of the 1,000 differences. That range is the **paired interval**.

Because both models always face the same proteins, the luck of the draw hits both alike and mostly cancels out of the difference. That makes the paired interval a fairer test than checking whether the two separate intervals overlap.

How to read it:
- **The whole interval is above zero:** on almost every re-draw, the fine-tuned model scored higher. The improvement is very unlikely to come from luck, so we call it real for that species. We say the interval *excludes zero*.
- **The whole interval is below zero:** the baseline is really better for that species. This interval also excludes zero.
- **The interval includes zero** (its low end is below zero and its high end is above): these test proteins cannot tell us which model is better. The difference we see might be luck.

Our helper `paired_differences(preds)` does this for every species. It returns a table with the columns `species`, `diff` (fine-tuned minus baseline, on the real test proteins), `low` and `high` (the two ends of the paired interval), and `excludes_zero` (True or False).

In [ ]:
# TODO: Compute the paired difference, fine-tuned minus baseline, for every species. Store the table in diffs.
# Hint: use paired_differences(preds); its defaults already compare the baseline with the fine-tuned model.
diffs = None  # replace None with your code

In [ ]:
# Your table: one row per species. Read the low and high columns together with excludes_zero.
diffs.round(3)

**Question 1.** For which species does the paired interval exclude zero? In each of those species, did fine-tuning make the score better or worse? What can you say, and not say, about the species whose interval includes zero?

**Your answer:**

## Where the mistakes go: confusion matrices
A **confusion matrix** shows where each model sends the proteins of each true class. Each row is one true class, and each column is one predicted class. Here each row is divided by its total, so a cell shows the *fraction* of that true class that went to that predicted class, and every row adds up to 1. Dark squares on the diagonal (true class equals predicted class) mean correct answers. Dark squares off the diagonal show which classes a model confuses with which.

Our helper `plot_confusion_grid(preds)` draws one matrix for every species (rows of the figure) and both real models (columns). The floor is left out, because it always predicts the same class. Every plotting helper in this project returns a matplotlib **figure**, and a figure's `.savefig(path, dpi=150, bbox_inches="tight")` saves it as an image: `dpi=150` sets the sharpness, and `bbox_inches="tight"` trims the empty border.

In [ ]:
# TODO: Draw the confusion matrices for every species and both models. Save the figure as confusion_grid.png
#       in the figures folder of your run folder, WORK_DIR / "figures".
# Hint: fig = plot_confusion_grid(...) returns the figure; then call fig.savefig(...) with the path
#       WORK_DIR / "figures" / "confusion_grid.png", dpi=150, and bbox_inches="tight".
# YOUR CODE HERE

The matrices show *where* the mistakes go. To compare classes with one number each, the next cell computes the F1 of every class, for every species and both models. Read down a column to follow one class from human to *E. coli*. An empty cell (`NaN`) means that species has no test proteins of that class.

In [ ]:
# per_class_f1 (from our helpers) returns one row per species, model, and class,
# with the columns species, model, ec_class, class_name, and f1.
class_f1 = per_class_f1(preds)
class_names = list(CLASS_NAMES.values())  # the 7 class names, in class order 1 to 7

# A small function that turns one model's rows into a grid: one row per species, one column per class.
def class_grid(model):
    rows = class_f1[class_f1["model"] == model]
    # pivot makes species the rows and class names the columns; reindex puts both in our usual order.
    return rows.pivot(index="species", columns="class_name", values="f1").reindex(index=SPECIES_ORDER, columns=class_names)

# display() shows a table the way the last line of a cell does, but it works anywhere in the cell.
print("Baseline: F1 of each class")
display(class_grid("baseline").round(2))
print("Fine-tuned: F1 of each class")
display(class_grid("finetuned").round(2))
print("Fine-tuned minus baseline (above 0 means fine-tuning scored higher on that class)")
display((class_grid("finetuned") - class_grid("baseline")).round(2))

## The headline figure
This is the figure that answers the project's question in one picture. The species go along the bottom in order of how long ago they shared an ancestor with humans, in millions of years ("Mya"). Each model is one line: a dot is its macro-F1 on one species, and the bar through the dot is that score's 95% interval. Our helper `load_divergence()` reads the divergence times from TimeTree, a public database of when species split apart.

In [ ]:
load_divergence()  # one row per species: millions of years since it shared an ancestor with humans, and the source

In [ ]:
# TODO: Draw the headline figure, macro-F1 against distance from human for the baseline and the fine-tuned model.
#       Save it as distance_curve.png in WORK_DIR / "figures".
# Hint: plot_distance_curve(results, load_divergence()) returns the figure; save it with .savefig as you did
#       for the confusion matrices.
# YOUR CODE HERE

Use this figure in your presentation. When the bars of the two models overlap, the figure alone cannot tell you whether fine-tuning helped for that species: check its paired interval in `diffs`. Notebook 05 asks you to explain the pattern you see here.

## Save the tables
Notebook 05 and your presentation use these tables, so the next cell saves them to your run folder.

In [ ]:
# Save as tab-separated files, as in notebooks 02 and 03. index=False leaves out pandas' row labels.
results.to_csv(WORK_DIR / "results.tsv", sep="\t", index=False)
diffs.to_csv(WORK_DIR / "paired_differences.tsv", sep="\t", index=False)
class_f1.to_csv(WORK_DIR / "per_class_f1.tsv", sep="\t", index=False)
print(f"Saved results.tsv, paired_differences.tsv, and per_class_f1.tsv to {WORK_DIR}")

## In your own words
**Question 2.** Which species has the widest intervals in your results table, and which the narrowest? What can make a bootstrap interval wide?

**Your answer:**

## Done
Your scores, tables, and figures are saved on your Drive.

**Check-in 2 comes next.** Your mentor will ask you to explain, in your own words and without AI help: how the fine-tuned model compares with the baseline; what macro-F1 measures and why we use it instead of accuracy; what the bootstrap interval and the paired interval are, and what it means when an interval excludes zero; and how the scores change from human to *E. coli*. Reread this notebook's explanations, your predictions, and your answers first.

After the check-in: notebook 05, where you look for the reasons behind these results.